## MiniProject IDS
### Network Intrusion Detection System using the UNSW-NB15 dataset 

## Part A
### The problem and the data

Every day, a large number of network requests and flows enter organizations. To be able to understand and react to the different types of network traffic, machine learning can be used. One way to do this is by creating an Intrusion Detection System (IDS) that can classify network flows as either malicious or benign. However, to train such a system, we need a large amount of labeled data, which can be difficult to obtain.

A machine-learning-based IDS can be used in companies and could assist Security Operations Centre (SOC) analysts and network administrators in detecting potential cyberattacks and malicious activity and potentially blocking harmful requests before they enter the system.

However, false alerts in an intrusion detection system can be costly, as security analysts monitoring these alerts need to investigate and determine whether they are actual attacks or not. Too many false alerts can lead to alert fatigue, where analysts become overwhelmed by the number of warnings. This can reduce the effectiveness of the system, as frequent false alerts can cause people to lose trust in it and potentially overlook real attacks. 


In [1]:
# Run in your terminal once: pip install -r requirements_ids.txt
from pathlib import Path
import warnings, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, recall_score, f1_score, precision_score,
                             average_precision_score, confusion_matrix)
from sklearn.base import clone
warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
START = time.perf_counter()
DATA_DIR = Path('data')
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
MAX_ROWS = 5000   # Use 4,000–5,000 for demonstration; mention sampling in report.
print('Data directory:', DATA_DIR.resolve())

Data directory: /Users/nishantgandhi/Library/CloudStorage/OneDrive-LuleåUniversityofTechnology/D7084E AI For Cybersecurity XAI/MiniProject/IntrusionDetectionSystem_MiniProject/data


## 1. Configure the experiment

In [2]:
project_dir = Path.cwd()
data_dir = project_dir / "data"
# Keep full-data runs for final results; sampling is useful when checking code quickly.
USE_SAMPLE = False
SAMPLE_FRAC = 0.25
csv_files = sorted(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files in data/")
for path in csv_files:
    print(" -", path.name)


Found 2 CSV files in data/
 - UNSW_NB15_testing-set.csv
 - UNSW_NB15_training-set.csv


## 2. Load and clean the CSV files

In [ ]:
frames = []
for csv_path in csv_files:
    try:
        frame = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        frame = pd.read_csv(csv_path, encoding="latin1", low_memory=False)
        print(f"Used Latin-1 fallback for {csv_path.name}")

    frame.columns = [str(col).strip() for col in frame.columns]

    if frame.empty:
        print(f"Skipping {csv_path.name}: empty file")
        continue

    # UNSW-NB15 uses a normal/attack category field and a numeric label field.
    label_col = next((col for col in frame.columns if col.lower() == "label"), None)
    attack_col = next((col for col in frame.columns if col.lower() == "attack_cat"), None)

    if label_col is None and attack_col is None:
        print(f"Skipping {csv_path.name}: no usable target column")
        continue

    if label_col is not None and label_col != "label":
        frame = frame.rename(columns={label_col: "label"})
    if attack_col is not None and attack_col != "attack_cat":
        frame = frame.rename(columns={attack_col: "attack_cat"})

    frames.append(frame)

if not frames:
    raise ValueError("No valid UNSW-NB15 CSV files were found in the data folder.")

# Combine the training and testing sets, keeping the normal-vs-attack binary target.
df = pd.concat(frames, ignore_index=True)

# Keep the core record identifier, but drop any metadata that is not useful for learning.
for col in ["id", "attack_cat"]:
    if col in df.columns:
        df = df.drop(columns=[col], errors="ignore")

# Clean numeric feature columns and convert the labelling into a binary target.
df = df.drop_duplicates().replace([np.inf, -np.inf], np.nan)
for col in df.columns:
    if col != "label":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# UNSW-NB15 uses 0 = normal, 1 = attack (and attack_cat shows the attack family).
# Some versions may carry an integer label already; the fallback handles both formats.
df["Label"] = df["label"].astype(str).str.strip().apply(
    lambda value: 0 if value.lower() in {"0", "normal", "benign"} else 1
)

# Keep the binary target for all later modelling work.
df = df.drop(columns=[col for col in ["label"] if col in df.columns], errors="ignore")

if USE_SAMPLE:
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED)

print("Cleaned dataset shape:", df.shape)
print("\nDataset columns:")
print(df.columns.tolist())
print("\nLabel distribution:")
print(df["Label"].value_counts().sort_index())

# Plot the binary class distribution for the UNSW-NB15 dataset.
label_counts = df["Label"].value_counts().sort_index()
plt.figure(figsize=(7, 5))
bars = plt.bar(["Normal", "Attack"], [label_counts.get(0, 0), label_counts.get(1, 0)])
plt.xlabel("Class")
plt.ylabel("Number of Flows")
plt.title("UNSW-NB15 Binary Class Distribution")
plt.ticklabel_format(style="plain", axis="y")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

# 60/20/20 split for the project requirements, stratified by the binary label.
X = df.drop(columns=["Label"])
y = df["Label"]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    train_size=0.6,
    stratify=y,
    random_state=SEED,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=SEED,
)

print("\nTrain/val/test split:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

majority_class = y_train.mode().iloc[0]
majority_acc = (y_train == majority_class).mean()
majority_recall = recall_score(y_test, np.full_like(y_test, majority_class, dtype=int))
print(f"\nTrivial baseline accuracy: {majority_acc:.4f}")
print(f"Trivial baseline recall: {majority_recall:.4f}")